# Chart QA Generation v2 - Research-Grade Dataset

| Version | Date | Author | Description |
| --- | --- | --- | --- |
| 2.0.0 | 2026-01-28 | That Le | Enhanced QA generation with reasoning depth |
| 2.1.0 | 2026-01-28 | That Le | Moved to data_factory (not a pipeline stage) |

---

## Overview

This notebook generates **research-grade QA pairs** for chart analysis with:

### Architecture Note

**QA Generation is NOT a pipeline stage.** It's a **data factory** tool for generating
SLM training data. The core pipeline is:

```
Stage 1 (Ingestion) -> Stage 2 (Detection) -> Stage 3 (Extraction) -> Stage 4 (Reasoning)
                                                                            |
                                                                     trained SLM
                                                                            ^
                                                        data_factory/qa_generator ----+
```

### Question Depth Levels

| Level | Types | Difficulty | Examples |
| --- | --- | --- | --- |
| **Shallow** | structural, extraction, counting | 1-2 | "What is the title?", "How many bars?" |
| **Intermediate** | comparison, trend, range | 2-3 | "Which is higher?", "Is it increasing?" |
| **Deep** | interpolation, % change, threshold | 3-5 | "Estimate value at X=0.35", "At what point does Y < 500?" |
| **Conceptual** | why_reasoning, caption_aware | 4-5 | "Why does trend change?", "What does caption imply?" |

### Features

1. **Visual Grounding**: References to specific chart regions, tick marks, data points
2. **Reasoning Steps**: Multi-step inference chains
3. **Confidence Scores**: Model confidence for each answer
4. **Uncertainty Handling**: "Cannot be determined" answers
5. **Structured Output**: JSON schema for training

## Setup

In [ ]:
import os
import sys
import json
import logging
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Any

# Add project root to path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# Load environment variables
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger(__name__)

# Paths
DATA_DIR = PROJECT_ROOT / "data" / "academic_dataset"
CLASSIFIED_DIR = DATA_DIR / "classified_charts"
QA_V2_DIR = DATA_DIR / "chart_qa_v2"
METADATA_DIR = DATA_DIR / "metadata"

print(f"Project root: {PROJECT_ROOT}")
print(f"QA v2 output: {QA_V2_DIR}")

In [ ]:
# Import QA Generator v2
from core_engine.schemas.qa_schemas import (
    QuestionType,
    ReasoningMethod,
    QAPairV2,
    ChartQASampleV2,
    QUESTION_TEMPLATES,
    QUESTION_DIFFICULTY,
)

# Display question type distribution target
print("Question Type Difficulty Mapping:")
print("=" * 40)
for qt, diff in sorted(QUESTION_DIFFICULTY.items(), key=lambda x: x[1]):
    print(f"  {qt.value:20s} -> Difficulty {diff}")

## 1. Load Sample Data

In [ ]:
# Load metadata for captions and context
metadata_cache = {}

for meta_file in METADATA_DIR.glob("*.json"):
    try:
        with open(meta_file, "r", encoding="utf-8") as f:
            data = json.load(f)
            # Index by image_id
            if isinstance(data, list):
                for item in data:
                    if "image_id" in item:
                        metadata_cache[item["image_id"]] = item
            elif "image_id" in data:
                metadata_cache[data["image_id"]] = data
    except Exception as e:
        logger.warning(f"Failed to load {meta_file}: {e}")

print(f"Loaded metadata for {len(metadata_cache)} images")

In [ ]:
# Build sample list from classified_charts
def build_sample_list(
    chart_types: List[str] = None,
    limit_per_type: int = None,
) -> List[Dict[str, Any]]:
    """Build list of samples to process."""
    samples = []
    
    if chart_types is None:
        chart_types = [d.name for d in CLASSIFIED_DIR.iterdir() if d.is_dir()]
    
    for chart_type in chart_types:
        type_dir = CLASSIFIED_DIR / chart_type
        if not type_dir.exists():
            continue
        
        images = list(type_dir.glob("*.png"))
        if limit_per_type:
            images = images[:limit_per_type]
        
        for img_path in images:
            image_id = img_path.stem
            meta = metadata_cache.get(image_id, {})
            
            samples.append({
                "image_id": image_id,
                "image_path": str(img_path),
                "chart_type": chart_type,
                "caption": meta.get("caption_text"),
                "context_text": meta.get("context_text"),
                "parent_paper_id": meta.get("parent_paper_id"),
            })
    
    return samples

# Test with small sample
test_samples = build_sample_list(
    chart_types=["line", "bar", "scatter"],
    limit_per_type=2,
)

print(f"Built {len(test_samples)} test samples")
for s in test_samples[:3]:
    print(f"  - {s['chart_type']}: {s['image_id']}")

## 2. Initialize QA Generator (Data Factory)

In [ ]:
# Import QA Generator from data_factory (NOT a pipeline stage)
from core_engine.data_factory import ChartQAGeneratorV2, QAGeneratorConfig

# Get API key
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY not found in environment")

# Create configuration with Gemini 3 Flash Preview
qa_config = QAGeneratorConfig(
    api_key=GEMINI_API_KEY,
    model_name="gemini-3-flash-preview",  # Latest model
    temperature=0.7,
    questions_per_chart=10,
    include_uncertain=False,
    output_dir=QA_V2_DIR / "generated",
    checkpoint_every=100,
)

# Initialize QA Generator
generator = ChartQAGeneratorV2(qa_config)

print("QA Generator initialized!")
print(f"  Model: {qa_config.model_name}")
print(f"  Output: {qa_config.output_dir}")
print(f"  Note: This is a DATA FACTORY tool, not a pipeline stage")

## 3. Test Single Image Generation

In [ ]:
# Test with a single line chart
test_sample = test_samples[0]

print(f"Testing with: {test_sample['image_id']}")
print(f"Chart type: {test_sample['chart_type']}")
print(f"Caption: {test_sample.get('caption', 'N/A')}")

# Display image
from PIL import Image
import matplotlib.pyplot as plt

img = Image.open(test_sample["image_path"])
plt.figure(figsize=(10, 8))
plt.imshow(img)
plt.axis("off")
plt.title(f"{test_sample['chart_type'].upper()}: {test_sample['image_id']}")
plt.show()

In [ ]:
# Generate QA pairs using generator
qa_sample = generator.generate_qa_pairs(
    image_path=test_sample["image_path"],
    chart_type=test_sample["chart_type"],
    caption=test_sample.get("caption"),
    context=test_sample.get("context_text"),
    metadata=test_sample,
)

print(f"\nGenerated {len(qa_sample.qa_pairs)} QA pairs")
print(f"Average difficulty: {qa_sample.avg_difficulty:.2f}")
print(f"\nQuestion type distribution:")
for qt, count in sorted(qa_sample.qa_distribution.items()):
    print(f"  {qt}: {count}")

In [ ]:
# Display generated QA pairs
print("=" * 80)
print("GENERATED QA PAIRS (v2)")
print("=" * 80)

for i, qa in enumerate(qa_sample.qa_pairs, 1):
    print(f"\n--- QA #{i} [{qa.question_type.value.upper()}] (Difficulty: {qa.difficulty}) ---")
    print(f"Q: {qa.question}")
    print(f"A: {qa.answer}")
    
    if qa.answer_value is not None:
        unit = f" {qa.answer_unit}" if qa.answer_unit else ""
        print(f"   Value: {qa.answer_value}{unit}")
    
    if qa.inference:
        print(f"   Method: {qa.inference.method.value}")
        print(f"   Confidence: {qa.inference.confidence:.2f} ({qa.inference.confidence_level.value})")
        
        if qa.inference.reasoning_steps:
            print(f"   Reasoning:")
            for step in qa.inference.reasoning_steps:
                print(f"     {step.step_number}. {step.action} -> {step.observation}")
    
    if qa.visual_grounding:
        regions = [r.value for r in qa.visual_grounding.regions_referenced]
        print(f"   Visual refs: {regions}")
        if qa.visual_grounding.tick_marks_used:
            print(f"   Tick marks: {qa.visual_grounding.tick_marks_used}")

## 4. Batch Generation (Full Dataset)

In [ ]:
# Configuration for full generation
GENERATION_CONFIG = {
    # Chart types to process (exclude 'uncertain' and 'other' for now)
    "chart_types": ["line", "bar", "scatter", "pie", "heatmap", "histogram", "area", "box"],
    
    # Limit per type (set to None for all)
    "limit_per_type": None,  # Change to None for full processing
    
    # Checkpoint frequency
    "checkpoint_every": 100,
    
    # Output directory
    "output_dir": QA_V2_DIR / "generated",
}

print("Generation Configuration:")
for k, v in GENERATION_CONFIG.items():
    print(f"  {k}: {v}")

In [ ]:
# Build full sample list
full_samples = build_sample_list(
    chart_types=GENERATION_CONFIG["chart_types"],
    limit_per_type=GENERATION_CONFIG["limit_per_type"],
)

print(f"\nTotal samples to process: {len(full_samples)}")
print("\nDistribution by chart type:")

type_counts = {}
for s in full_samples:
    ct = s["chart_type"]
    type_counts[ct] = type_counts.get(ct, 0) + 1

for ct, count in sorted(type_counts.items(), key=lambda x: -x[1]):
    pct = count / len(full_samples) * 100
    print(f"  {ct:12s}: {count:6d} ({pct:5.1f}%)")

In [ ]:
# CAUTION: This will call Gemini API for all samples!
# Uncomment to run full generation

RUN_FULL_GENERATION = False  # Set to True to run

if RUN_FULL_GENERATION:
    print(f"Processing {len(full_samples)} samples...")
    
    # Use generator to process all samples
    all_results = generator.generate_batch(
        samples=full_samples,
        output_dir=GENERATION_CONFIG["output_dir"],
        checkpoint_every=GENERATION_CONFIG["checkpoint_every"],
    )
    
    print(f"\nGeneration complete!")
    print(f"  Total samples: {len(all_results)}")
    print(f"  Total QA pairs: {sum(len(r.qa_pairs) for r in all_results)}")
    print(f"  Output: {GENERATION_CONFIG['output_dir']}")

## 5. Convert to Sharded Format

In [ ]:
def create_shards_v2(
    qa_samples: List[ChartQASampleV2],
    output_dir: Path,
    shard_size: int = 1000,
) -> Dict[str, Any]:
    """Create sharded dataset from QA samples."""
    from collections import defaultdict
    
    output_dir = Path(output_dir)
    shards_dir = output_dir / "shards_v2"
    shards_dir.mkdir(parents=True, exist_ok=True)
    
    # Group by chart type
    by_type = defaultdict(list)
    for sample in qa_samples:
        by_type[sample.chart_type].append(sample)
    
    # Statistics
    total_images = len(qa_samples)
    total_qa_pairs = sum(len(s.qa_pairs) for s in qa_samples)
    
    # Question type distribution
    qa_type_dist = defaultdict(int)
    difficulty_dist = defaultdict(int)
    
    for sample in qa_samples:
        for qa in sample.qa_pairs:
            qa_type_dist[qa.question_type.value] += 1
            difficulty_dist[qa.difficulty] += 1
    
    # Create shards
    shard_files = []
    
    for chart_type, samples in by_type.items():
        type_dir = shards_dir / chart_type
        type_dir.mkdir(exist_ok=True)
        
        for i in range(0, len(samples), shard_size):
            shard_samples = samples[i:i + shard_size]
            shard_index = i // shard_size
            
            shard_path = type_dir / f"shard_{shard_index:03d}.json"
            
            shard_data = {
                "schema_version": "2.0.0",
                "chart_type": chart_type,
                "shard_index": shard_index,
                "sample_count": len(shard_samples),
                "samples": [s.model_dump() for s in shard_samples],
            }
            
            with open(shard_path, "w", encoding="utf-8") as f:
                json.dump(shard_data, f, indent=2, ensure_ascii=False, default=str)
            
            shard_files.append(str(shard_path.relative_to(output_dir)))
    
    # Write metadata
    metadata = {
        "schema_version": "2.0.0",
        "created_at": datetime.now().isoformat(),
        "description": "Research-grade Chart QA dataset with reasoning depth",
        "statistics": {
            "total_images": total_images,
            "total_qa_pairs": total_qa_pairs,
            "avg_qa_per_image": total_qa_pairs / total_images if total_images else 0,
            "chart_type_distribution": {k: len(v) for k, v in by_type.items()},
            "question_type_distribution": dict(qa_type_dist),
            "difficulty_distribution": dict(difficulty_dist),
        },
        "shard_files": shard_files,
    }
    
    meta_path = output_dir / "metadata_v2.json"
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)
    
    print(f"Created {len(shard_files)} shard files")
    print(f"Metadata saved to: {meta_path}")
    
    return metadata

## 6. Quality Analysis

In [ ]:
def analyze_qa_quality(samples: List[ChartQASampleV2]) -> Dict[str, Any]:
    """Analyze quality metrics for generated QA pairs."""
    from collections import defaultdict
    import numpy as np
    
    metrics = {
        "total_samples": len(samples),
        "total_qa_pairs": 0,
        "by_question_type": defaultdict(int),
        "by_difficulty": defaultdict(int),
        "by_confidence_level": defaultdict(int),
        "by_reasoning_method": defaultdict(int),
        "answerable_count": 0,
        "unanswerable_count": 0,
        "with_reasoning_steps": 0,
        "confidences": [],
        "difficulties": [],
    }
    
    for sample in samples:
        for qa in sample.qa_pairs:
            metrics["total_qa_pairs"] += 1
            metrics["by_question_type"][qa.question_type.value] += 1
            metrics["by_difficulty"][qa.difficulty] += 1
            metrics["difficulties"].append(qa.difficulty)
            
            if qa.is_answerable:
                metrics["answerable_count"] += 1
            else:
                metrics["unanswerable_count"] += 1
            
            if qa.inference:
                metrics["by_confidence_level"][qa.inference.confidence_level.value] += 1
                metrics["by_reasoning_method"][qa.inference.method.value] += 1
                metrics["confidences"].append(qa.inference.confidence)
                
                if qa.inference.reasoning_steps:
                    metrics["with_reasoning_steps"] += 1
    
    # Calculate aggregates
    if metrics["confidences"]:
        metrics["avg_confidence"] = np.mean(metrics["confidences"])
        metrics["std_confidence"] = np.std(metrics["confidences"])
    
    if metrics["difficulties"]:
        metrics["avg_difficulty"] = np.mean(metrics["difficulties"])
    
    # Clean up lists for display
    del metrics["confidences"]
    del metrics["difficulties"]
    
    return metrics

# Example usage (with test sample)
if 'qa_sample' in dir():
    test_metrics = analyze_qa_quality([qa_sample])
    print("Quality Metrics (test sample):")
    print(json.dumps(test_metrics, indent=2, default=str))

## 7. Export for Training

In [ ]:
def export_for_training(
    samples: List[ChartQASampleV2],
    output_path: Path,
    format: str = "jsonl",
) -> int:
    """Export QA pairs in training format.
    
    Formats:
    - jsonl: One QA pair per line (for fine-tuning)
    - conversation: Multi-turn conversation format
    """
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    
    count = 0
    
    with open(output_path, "w", encoding="utf-8") as f:
        for sample in samples:
            for qa in sample.qa_pairs:
                record = {
                    "image_id": sample.image_id,
                    "image_path": sample.image_path,
                    "chart_type": sample.chart_type,
                    "question": qa.question,
                    "answer": qa.answer,
                    "question_type": qa.question_type.value,
                    "difficulty": qa.difficulty,
                    "answer_value": qa.answer_value,
                    "is_answerable": qa.is_answerable,
                }
                
                # Add reasoning info if available
                if qa.inference:
                    record["reasoning_method"] = qa.inference.method.value
                    record["confidence"] = qa.inference.confidence
                
                f.write(json.dumps(record, ensure_ascii=False) + "\n")
                count += 1
    
    print(f"Exported {count} QA pairs to {output_path}")
    return count

## Summary

### QA Schema v2 Features

| Feature | Description |
| --- | --- |
| **Question Types** | 16 types from shallow to conceptual |
| **Difficulty Levels** | 1-5 scale with clear definitions |
| **Visual Grounding** | References to chart regions, tick marks |
| **Reasoning Steps** | Multi-step inference chains |
| **Confidence Scores** | 0-1 confidence with categorical levels |
| **Uncertainty** | Explicit "cannot be determined" handling |

### Target Distribution

| Category | Target % | Question Types |
| --- | --- | --- |
| Shallow | 20-30% | structural, extraction, counting |
| Intermediate | 30-40% | comparison, trend, range |
| Deep | 30-40% | interpolation, % change, threshold, multi-hop |
| Conceptual | 10-15% | why_reasoning, caption_aware |